# Agentic Workflow structure 1

## Corrective RAG
> CRAG는 LLM이 생성한 답변의 **신뢰도(Fidelity)**를 평가하여, 신뢰도가 낮을 경우 **검색 전략을 수정(Corrective)**하고 재시도하는 루프를 핵심으로 합니다.

즉, 신뢰도가 낮을 경우 다시 retrival하는 것임. 우리가 이 평가하는 구조를 RAG 워크플로우 안에서 정의해주는 것이다. 

### Corrective RAG LangGraph 구현 단계별 정리

 일반 RAG가 `검색 -> 생성`으로 끝나는 구조라면, CRAG는 생성된 답변의 신뢰도를 평가한 뒤 검색 전략을 수정하면서 다시 검색하거나 종료한다. 따라서 핵심은 답변 품질에 따라 그래프의 다음 경로를 바꾸는 조건부 라우팅이다.

### 1. 기본 구성 요소 준비

먼저 Gemini LLM, Gemini Embedding, Chroma 벡터 저장소, retriever를 준비한다. Chroma에는 CRAG 관련 더미 문서가 저장되어 있고, `retriever`는 질문과 관련된 문서를 검색하는 역할을 한다.

```python
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0)
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
db = Chroma.from_texts([...], embeddings)
retriever = db.as_retriever(search_kwargs={"k": 3})
```

이후 LangGraph는 이 retriever와 LLM을 노드 내부에서 호출하면서 CRAG 흐름을 실행한다.

### 2. CRAG 상태 스키마 정의

`CRAGState`는 그래프 전체에서 공유되는 상태이다. LangGraph의 각 노드는 이 상태를 입력으로 받고, 변경할 값만 dict로 반환해서 상태를 업데이트한다.

- `question`: 사용자의 질문
- `context`: 검색된 문서 리스트
- `answer`: LLM이 생성한 답변
- `generation_attempts`: 답변 생성 시도 횟수
- `search_strategy`: 현재 검색 전략

`search_strategy`는 `Literal["standard", "rewrite", "web_search"]`로 제한되어 있다. 즉, 검색 전략은 기본 검색, 쿼리 재작성, 웹 검색 방식 중 하나만 가질 수 있다.

또한 `generation_attempts`에는 `Annotated[int, add]`가 사용된다. 이 설정 때문에 노드가 `{"generation_attempts": 1}`을 반환하면 기존 값에 1이 누적된다.

### 3. `retrieve` 노드

`retrieve` 노드는 현재 검색 전략에 따라 검색 쿼리를 만든 뒤 retriever를 실행한다.

- `standard`: 원래 질문 그대로 검색
- `rewrite`: 질문 앞에 `재작성_쿼리:`를 붙여 재작성 전략을 흉내냄(이 글에서는 구체적으로 구현하지 않음)
- `web_search`: 질문 앞에 `웹_검색_쿼리:`를 붙여 외부 검색 전략을 흉내냄(이 글에서는 구체적으로 구현하지 않음)

검색된 문서는 `context`에 저장된다. 이 노드는 답변을 만들지 않고, 다음 `generate` 노드가 사용할 근거 문서만 준비한다.

### 4. `generate_answer` 노드

`generate_answer` 노드는 검색된 `context`를 문자열로 합친 뒤 LLM에게 질문과 함께 전달한다. 프롬프트에는 답변 끝에 `[Fidelity: High/Medium/Low]` 중 하나를 명시하라는 지시가 들어 있다.

이 Fidelity 값은 이후 라우터가 답변의 신뢰도를 판단하는 기준으로 사용한다.

노드가 반환하는 값은 다음과 같다.

```python
return {"answer": response.content, "generation_attempts": 1}
```

즉, 답변을 상태에 저장하고 생성 시도 횟수를 1 증가시킨다.

### 5. `update_strategy` 노드

`update_strategy`는 다음 검색에 사용할 전략을 바꾸는 단순한 상태 업데이트 함수이다. 그래프에는 이 함수를 직접 하나만 넣는 대신, 람다로 감싸서 두 개의 노드로 등록한다.

```python
crag_workflow.add_node("update_rewrite", lambda x: update_strategy(x, "rewrite"))
crag_workflow.add_node("update_web", lambda x: update_strategy(x, "web_search"))
```

`update_rewrite`는 검색 전략을 `rewrite`로 바꾸고, `update_web`은 검색 전략을 `web_search`로 바꾼다. 전략을 바꾼 뒤에는 다시 `retrieve`로 돌아가 재검색한다.

### 6. `grade_and_adjust_router` 라우터

`grade_and_adjust_router`는 LangGraph의 노드가 아니라 조건부 엣지에서 사용하는 라우터 함수이다. 노드처럼 상태를 업데이트하는 것이 아니라, 다음 이동 경로를 나타내는 문자열을 반환한다.

라우터는 먼저 답변 안의 `[Fidelity: ...]` 값을 파싱한다.

- `High`: 답변 신뢰도가 충분하므로 `pass` 반환
- 최대 시도 횟수 초과: 더 이상 재시도하지 않고 `fail` 반환
- 현재 전략이 `standard`: `rewrite` 반환
- 현재 전략이 `rewrite`: `web_search` 반환
- `web_search`까지 실패: `fail` 반환

즉, CRAG의 핵심 보정 로직은 이 라우터에 들어 있다. 답변이 좋으면 종료하고, 부족하면 검색 전략을 점점 강화한다.

### 7. LangGraph 그래프 구성

그래프는 `StateGraph(CRAGState)`로 생성한다.

```python
crag_workflow = StateGraph(CRAGState)
```

그다음 실제 상태를 업데이트하는 함수들을 노드로 등록한다.

```python
crag_workflow.add_node("retrieve", retrieve)
crag_workflow.add_node("generate", generate_answer)
crag_workflow.add_node("update_rewrite", lambda x: update_strategy(x, "rewrite"))
crag_workflow.add_node("update_web", lambda x: update_strategy(x, "web_search"))
```

시작점은 `retrieve`이다. 즉, 사용자의 질문이 들어오면 먼저 검색부터 수행한다.

```python
crag_workflow.set_entry_point("retrieve")
crag_workflow.add_edge("retrieve", "generate")
```

검색이 끝나면 항상 `generate`로 이동한다. 그 뒤에는 답변 신뢰도에 따라 조건부 분기가 일어난다.

```python
crag_workflow.add_conditional_edges(
    "generate",
    grade_and_adjust_router,
    {
        "pass": END,
        "rewrite": "update_rewrite",
        "web_search": "update_web",
        "fail": END,
    }
)
```

전략 업데이트 노드는 다시 `retrieve`로 연결된다.

```python
crag_workflow.add_edge("update_rewrite", "retrieve")
crag_workflow.add_edge("update_web", "retrieve")
```

따라서 전체 흐름은 다음과 같다.

```text
retrieve -> generate -> grade_and_adjust_router
                         -> pass -> END
                         -> rewrite -> update_rewrite -> retrieve
                         -> web_search -> update_web -> retrieve
                         -> fail -> END
```

### 8. 컴파일과 실행

모든 노드와 엣지를 정의한 뒤 `compile()`을 호출하면 실행 가능한 LangGraph 앱이 만들어진다.

```python
crag_app = crag_workflow.compile()
```

실행할 때는 초기 상태를 dict로 넣는다.

```python
initial_state_crag = {
    "question": "LangGraph는 CRAG에 왜 적합하며, CRAG의 주요 장점은 무엇인가요?",
    "context": [],
    "answer": "",
    "generation_attempts": 0,
    "search_strategy": "standard"
}

final_state_crag = crag_app.invoke(initial_state_crag)
```

그래프는 초기 전략인 `standard`로 검색을 수행하고, 답변을 생성한 뒤 Fidelity를 기준으로 종료하거나 재검색한다.

## 핵심 요약

이 코드는 Corrective RAG를 LangGraph로 구현한 구조이다. 핵심은 단순히 문서를 검색하고 답변을 생성하는 것이 아니라, 생성된 답변의 신뢰도를 평가한 뒤 검색 전략을 바꾸며 다시 시도할 수 있다는 점이다.

`retrieve`는 검색을 담당하고, `generate`는 답변 생성을 담당한다. `grade_and_adjust_router`는 답변의 Fidelity와 시도 횟수를 보고 다음 경로를 결정한다. 신뢰도가 높으면 종료하고, 부족하면 `rewrite` 전략으로 재검색하며, 그래도 부족하면 `web_search` 전략으로 다시 검색한다. 최대 시도 횟수를 넘거나 마지막 전략에서도 실패하면 종료한다.

LangGraph를 사용하면 이런 흐름을 노드와 엣지로 명확하게 표현할 수 있다. 특히 CRAG처럼 `검색 -> 생성 -> 평가 -> 전략 수정 -> 재검색`이 반복되는 에이전트 워크플로우에서는 상태 기반 그래프 구조가 일반적인 체인보다 제어하기 쉽고, 각 단계의 역할을 분리하기 좋다.

In [ ]:
import os
from typing import TypedDict, List, Annotated, Literal
from operator import add
from langchain_core.documents import Document
from langchain_core.runnables import Runnable
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import AIMessage, HumanMessage
from langgraph.graph import StateGraph, END

# --- 기본 셋업 ---
# os.environ["GOOGLE_API_KEY"] = "YOUR_GEMINI_API_KEY"

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0)
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
db = Chroma.from_texts(
    ["CRAG (Corrective RAG)는 답변 검증 후 검색 전략을 조정하는 기법이다.",
    "CRAG는 신뢰도 평가를 통해 쿼리 재작성 또는 외부 검색을 결정한다."],
    embeddings
)
retriever: Runnable = db.as_retriever(search_kwargs={"k": 3})

# --- CRAG 상태 정의 및 상수 ---
class CRAGState(TypedDict):
    """CRAG 워크플로우의 공유 상태"""
    question: str
    context: List[Document]
    answer: str
    generation_attempts: Annotated[int, add] # 재시도 횟수
    search_strategy: Literal["standard", "rewrite", "web_search"] # 현재 검색 전략 # Literal을 사용하면 search_strategy에 올 수 있는 값을  "standard", "rewrite", "web_search"로 제한할 수 있다. 
    
MAX_ATTEMPTS = 3 

# --- A. 노드 함수 정의 (dict 반환) ---

def retrieve(state: CRAGState) -> CRAGState:
    """문서를 검색하고 상태를 업데이트합니다."""
    current_strategy = state.get("search_strategy", "standard")
    print(f"--- 🔍 CRAG: {current_strategy} 전략으로 검색 수행 ---")
    
    query = state["question"]
    if current_strategy == "rewrite":
        query = f"재작성_쿼리: {query}"
    elif current_strategy == "web_search":
        query = f"웹_검색_쿼리: {query}"

    docs = retriever.invoke(query)
    # 📌 노드는 dict를 반환하여 상태를 업데이트합니다.
    return {"context": docs}

def generate_answer(state: CRAGState) -> CRAGState:
    """답변을 생성하고 시도 횟수를 업데이트합니다."""
    print("--- 🧠 CRAG: 답변 생성 중 ---")
    
    context_str = "\n\n".join([doc.page_content for doc in state["context"]])
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", "당신은 CRAG 에이전트입니다. Context를 기반으로 질문에 답하고, 답변이 Context에 충분히 근거하는지 스스로 판단하여 답변 끝에 **[Fidelity: High/Medium/Low]** 중 하나를 명시하십시오. Context가 매우 부족하면 'Low'로 표시하십시오."),
        ("human", "Context:\n{context}\n\nQuestion: {question}"),
    ])
    
    response = llm.invoke(prompt.format_messages(context=context_str, question=state["question"]))
    
    # 📌 노드는 dict를 반환합니다. generation_attempts가 1 증가합니다.
    return {"answer": response.content, "generation_attempts": 1}

def update_strategy(state: CRAGState, new_strategy: str) -> CRAGState:
    """다음 검색을 위한 전략을 상태에 설정합니다."""
    # 📌 노드는 dict를 반환합니다.
    return {"search_strategy": new_strategy}

# --- B. 라우터 함수 정의 (문자열 반환) ---

def grade_and_adjust_router(state: CRAGState) -> str:
    """
    라우터 함수: 답변의 신뢰도를 평가하고, 다음 노드 이름(문자열)을 반환합니다.
    이 함수는 LangGraph에서 노드로 추가되지 않고, 엣지(Edge)의 조건으로 사용됩니다.
    """
    print("--- 🧐 CRAG 라우터: 라우팅 결정 ---")
    
    answer = state["answer"]
    attempts = state["generation_attempts"]
    current_strategy = state["search_strategy"]
    
    # 1. 신뢰도 레벨 파싱
    fidelity = "Low"
    if "[Fidelity:" in answer:
        try:
            fidelity_part = answer.split("[Fidelity:")[1].split("]")[0].strip().lower()
            if "high" in fidelity_part:
                fidelity = "High"
            elif "medium" in fidelity_part:
                fidelity = "Medium"
        except IndexError:
            pass
            
    print(f"   -> 평가 신뢰도: {fidelity}, 현재 전략: {current_strategy}, 시도 횟수: {attempts}")

    # 2. 신뢰도 및 시도 횟수 기반 라우팅 로직
    if fidelity == "High":
        return "pass" 
    
    if attempts >= MAX_ATTEMPTS:
        return "fail"
    
    if current_strategy == "standard":
        return "rewrite"
    
    elif current_strategy == "rewrite":
        return "web_search"
    
    return "fail" # web_search 전략으로도 실패한 경우

In [ ]:
# --- 그래프 빌드 ---
crag_workflow = StateGraph(CRAGState)

# 1. 노드 추가 (dict를 반환하는 함수만 추가)
crag_workflow.add_node("retrieve", retrieve)
crag_workflow.add_node("generate", generate_answer)

# update_strategy 함수를 래핑하여 특정 전략을 설정하는 노드 생성
crag_workflow.add_node("update_rewrite", lambda x: update_strategy(x, "rewrite"))
crag_workflow.add_node("update_web", lambda x: update_strategy(x, "web_search"))


# 2. 엣지 정의 및 라우팅 (grade_and_adjust_router 사용)
crag_workflow.set_entry_point("retrieve")
crag_workflow.add_edge("retrieve", "generate")
# generate 다음은 라우터(grade_and_adjust_router)를 통해 조건부 분기
crag_workflow.add_conditional_edges(
    "generate",
    grade_and_adjust_router, # 👈 오류를 유발했던 함수를 라우터로 사용
    {
        "pass": END,
        "rewrite": "update_rewrite",
        "web_search": "update_web",
        "fail": END,
    }
)

# 전략 업데이트 노드들은 다시 검색 노드로 루프백
crag_workflow.add_edge("update_rewrite", "retrieve")
crag_workflow.add_edge("update_web", "retrieve")

crag_app = crag_workflow.compile()
print("\n✅ CRAG Workflow 수정 완료 및 컴파일됨.")
# 그래프 시각화가 매우 중요하다. 노드들을 잘못 연결하는 경우가 많기 때문이다. 그러므로 내가 생각하는 그래프 구조와 실제 시각화 과정이 맞는지 검토하는 과정이 매우 중요하다. 

# 결국 이 구조는 나의 질문과 답변이 얼만큼 신뢰도가 있는 지에 따라 어떤 전략을 사용할지, 단순 retrieve하고 끝날지 아니면 다른 구조로 갈 지를 정하는 구조를 corective RAG라고 한다. 

In [ ]:
# --- 실행 예시 ---
initial_state_crag = {"question": "LangGraph는 CRAG에 왜 적합하며, CRAG의 주요 장점은 무엇인가요?", "context": [], "answer": "", "generation_attempts": 0, "search_strategy": "standard"}
print("--- 🚀 CRAG 워크플로우 실행 시작 ---")
final_state_crag = crag_app.invoke(initial_state_crag)
print(f"\n--- ✅ CRAG 최종 답변 ---\n{final_state_crag['answer']}")

In [ ]:
fail_state_crag = {"question": "최신 AI 뉴스에 대해서 알려줘", "context": [], "answer": "", "generation_attempts": 0, "search_strategy": "standard"}

fail_state_crag = crag_app.invoke(fail_state_crag)
print(f"\n--- ✅ CRAG 최종 답변 ---\n{fail_state_crag['answer']}")